# Tutorial 2: From Sine Waves to the Fourier Series

**Week 2, Day 2: Signal Processing**

**By Neuromatch Academy (community contribution)**

**Content Creators:** *to be filled in*

**Content Reviewers:** *to be filled in*

**Production editors:** *to be filled in*

---


# Tutorial Objectives

*Estimated timing of tutorial: to be filled in*

This tutorial builds the core intuition behind Fourier analysis from the ground up. We start with the **sine wave** — the elementary building block of every signal — and learn how its three parameters (amplitude, frequency, phase) shape it. We then **decompose** a signal into a sum of sine waves, seeing how a handful of weighted components combine to reproduce a target. Finally, we use this idea to build the **Fourier series**: writing familiar periodic waveforms (square and sawtooth) as sums of harmonically related sines, watching the approximation converge as we add more terms, and even building an *aperiodic* Gaussian pulse the same way — where the **phase** alone decides whether the result is a sharp spike or noise.

By the end of this tutorial you will be able to:

- Describe a sine wave in terms of amplitude, frequency, and phase
- Reconstruct a target signal as a weighted sum of sine waves plus a DC offset
- Read a magnitude spectrum and relate it to the time-domain components
- Explain how negative coefficients correspond to a phase shift of $\pi$
- Build square and sawtooth waves from their Fourier components
- Reason about how the rate of coefficient decay controls convergence (and the Gibbs phenomenon)
- See how an aperiodic pulse is built from sine waves, and how phase alone separates a sharp spike from noise

---


# Setup

## Install and import feedback gadget

In [1]:
# @title Install and import feedback gadget
!pip3 install vibecheck datatops --quiet
from vibecheck import DatatopsContentReviewContainer
def content_review(notebook_section: str):
    return DatatopsContentReviewContainer(
        "",
        notebook_section,
        {
            "url": "https://pmyvdlilci.execute-api.us-east-1.amazonaws.com/klab",
            "name": "neuromatch_cn",
            "user_key": "y1x3mpx5",
        },
    ).render()
feedback_prefix = "W2D2_T2"

## Imports

In [2]:
# @title Imports
import io
import time
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML
import warnings
warnings.filterwarnings('ignore')

## Figure settings

In [3]:
# @title Figure Settings
import logging
logging.getLogger('matplotlib.font_manager').disabled = True
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
plt.style.use("https://raw.githubusercontent.com/NeuromatchAcademy/course-content/main/nma.mplstyle")

## Helper functions

In [ ]:
# @title Helper Functions

# Each interactive demo in this tutorial is fully self-contained (its config,
# time base, and callbacks live inside its own function), so no shared helper
# functions are required here.

---

# Section 1: The Sine Wave

*Estimated timing to here from start of tutorial: min*


## Video 1: The sine wave

In [4]:
from IPython.display import HTML
HTML("<div style='background:#eee;padding:1em;border-radius:6px;'>"
     "<b>Video 1 placeholder.</b> Topics: amplitude, frequency and phase; why the sine wave is the elementary building block of signals."
     "</div>")

In [5]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Video_1_SineWave")

A sine wave is the most basic building block of signals. It is described by three parameters:

| Parameter | Symbol | What it controls |
|-----------|--------|------------------|
| **Amplitude** | $A$ | Height of the wave (how loud / how strong) |
| **Frequency** | $f$ | How many cycles per second (measured in Hz) |
| **Phase** | $\varphi$ | Where in its cycle the wave starts (in radians) |

Together they give:

$$s(t) = A \cdot \sin(2\pi f t + \varphi)$$

A **mystery signal** is shown in the interactive demo below (gray dashed line). Your task is to match it by tuning the three sliders. Watch how each slider changes the shape of your signal (blue), and try to drive the **MSE as close to 0 as possible**.

## Coding Exercise 1: Implement the sine wave

Before you play with the sliders, write the sine-wave equation yourself.

Fill in `make_sine_wave(t, A, f, phi)` below so that it returns

$$s(t) = A \cdot \sin(2\pi f t + \varphi)$$

evaluated at every time sample in `t`. The interactive demo further down the page calls this function to draw your reconstruction (blue) — so once you implement it correctly, the sliders will spring to life.

In [6]:
def make_sine_wave(t, A, f, phi):
    """Generate a sine wave.

    Args:
        t (ndarray): time samples in seconds, shape (n_samples,)
        A (float):   amplitude
        f (float):   frequency in Hz
        phi (float): phase offset in radians

    Returns:
        ndarray: s(t) = A * sin(2π f t + φ), shape (n_samples,)
    """
    ####################################################
    ## TODO for students: implement the sine-wave equation.
    # Fill out the function and remove the line below.
    #raise NotImplementedError("Student exercise: implement the sine wave equation")
    ####################################################

    # Compute the sine wave  s(t) = A · sin(2π f t + φ)
    s = A * np.sin(2 * np.pi * f * t + phi)

    return s

[*Click for solution*](https://github.com/NeuromatchAcademy/course-content/tree/main/tutorials/W2D2_SignalProcessing/solutions/W2D2_Tutorial2_Solution_make_sine_wave.py)

In [7]:
# @title Solution

def make_sine_wave(t, A, f, phi):
    """Generate a sine wave: s(t) = A * sin(2π f t + φ)."""
    s = A * np.sin(2 * np.pi * f * t + phi)
    return s

In [8]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Exercise_1_SineWave")

In [9]:
# @title Interactive Demo 1: Match the target sine wave

def build_sine_demo():
    # ── Instructor config ──
    A_TRUE, F_TRUE, PHI_TRUE = 1.5, 5.0, 1.0
    DURATION, FS = 1.0, 1000
    t = np.linspace(0, DURATION, int(FS * DURATION), endpoint=False)
    y_true = A_TRUE * np.sin(2 * np.pi * F_TRUE * t + PHI_TRUE)

    amp_slider = widgets.FloatSlider(
        value=1.0, min=0.0, max=3.0, step=0.05,
        description='Amplitude (A)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='420px')
    )

    freq_slider = widgets.FloatSlider(
        value=1.0, min=1.0, max=20.0, step=0.5,
        description='Frequency f (Hz)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='420px')
    )

    phase_slider = widgets.FloatSlider(
        value=0.0, min=0.0, max=2 * np.pi, step=0.05,
        description='Phase φ (rad)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='420px'),
        readout_format='.2f'
    )

    img_out = widgets.Image(format='png')


    def update(_change=None):
        A   = amp_slider.value
        f   = freq_slider.value
        phi = phase_slider.value

        y_student = make_sine_wave(t, A, f, phi)
        mse = np.mean((y_true - y_student) ** 2)

        fig, ax = plt.subplots(figsize=(9, 4))

        ax.plot(t, y_true, color='#888888', linestyle='--', linewidth=2,
                label='Target signal', alpha=0.85)
        ax.plot(t, y_student, color='#2563EB', linewidth=2,
                label=f'Your signal  (A={A:.2f}, f={f:.1f} Hz, φ={phi:.2f} rad)')

        ax.set_title('Single Sine Wave — Match the Target', fontsize=14)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Amplitude')
        ax.legend(loc='upper right', fontsize=9)
        ax.set_xlim(0, DURATION)
        ax.set_ylim(-3.5, 3.5)

        mse_color = '#16a34a' if mse < 0.01 else '#dc2626'
        ax.text(
            0.02, 0.95, f'MSE: {mse:.4f}',
            transform=ax.transAxes, fontsize=12, verticalalignment='top',
            color=mse_color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor=mse_color, alpha=0.8)
        )

        fig.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img_out.value = buf.read()


    amp_slider.observe(update, names='value')
    freq_slider.observe(update, names='value')
    phase_slider.observe(update, names='value')

    ui = widgets.VBox([
        amp_slider,
        freq_slider,
        phase_slider,
        img_out
    ])

    display(ui)
    update()


build_sine_demo()

In [10]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Demo_1_SineMatch")

---
## Anatomy of a Sine Wave

$$\boxed{s(t) = A \cdot \sin(2\pi f t + \varphi)}$$

### Amplitude $A$
Controls the **peak value** of the wave. Doubling $A$ doubles the height without changing the speed or timing of the oscillation. In physical systems, amplitude often relates to energy: signal power scales as $A^2$.

### Frequency $f$
Controls how many **full cycles** occur per second (unit: Hz = cycles/second). A higher frequency means a faster oscillation and a shorter period:

$$T = \frac{1}{f}$$

The factor $2\pi f$ converts from cycles/second to **angular frequency** $\omega = 2\pi f$ (radians/second), because one full cycle spans $2\pi$ radians.

### Phase $\varphi$
Controls the **starting position** of the oscillation at $t = 0$ (unit: radians). A phase of $\varphi = \pi/2$ shifts the sine into a cosine; $\varphi = \pi$ flips it upside down. Phase only matters *relative* to another signal — a single isolated sine wave with any phase looks the same in the frequency domain.

### Key intuitions
- Moving the **amplitude slider** stretches the wave **vertically**.
- Moving the **frequency slider** compresses the wave **horizontally** (more wiggles per second).
- Moving the **phase slider** **slides** the wave left or right in time.

---

# Section 2: Signal Decomposition

*Estimated timing to here from start of tutorial: min*


## Video 2: Decomposing a signal into sine waves

In [11]:
from IPython.display import HTML
HTML("<div style='background:#eee;padding:1em;border-radius:6px;'>"
     "<b>Video 2 placeholder.</b> Topics: summing weighted sine waves, the DC offset, and the magnitude spectrum."
     "</div>")

In [12]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Video_2_Decomposition")

A mystery signal has been constructed from a constant offset plus two sine waves:

$$s(t) = w_0 + w_1 \cdot \sin(2\pi f_1 t) + w_2 \cdot \sin(2\pi f_2 t)$$

You know the frequencies — **$f_1$ = 3 Hz** and **$f_2$ = 7 Hz** — but the weights $w_0$, $w_1$ and $w_2$ are hidden.

**Your goal:** use the sliders in the interactive demo to adjust the weights until your reconstructed signal (blue) matches the ground truth (gray dashed) as closely as possible. The quality of your match is measured by the **Mean Squared Error (MSE)** shown on the plot — lower is better. Try to drive it as close to **0** as you can!

### A note on weights

The weights here are *amplitudes* — they represent how much of each frequency component is present in the signal. Each weight independently controls the contribution of its frequency.

$w_0$ is the **DC offset** — the **0 Hz component**, i.e. the constant average level the whole signal sits on. It is a Fourier coefficient just like the others, and you tune it with its own slider.

## Coding Exercise 2: Build the composite signal

Before you tune the sliders, write the equation for the composite signal yourself.

Fill in `make_signal(t, w0, w1, w2, f1, f2, phi1, phi2)` below so that it returns

$$S(t) = w_0 + w_1 \sin(2\pi f_1 t + \varphi_1) + w_2 \sin(2\pi f_2 t + \varphi_2)$$

evaluated at every time sample in `t`. The phase arguments default to `0`, so for the no-phase case you only need the DC offset plus two sines. The interactive demo below calls this function to draw your reconstruction (blue) — once you implement it, the sliders come to life.

In [13]:
def make_signal(t, w0, w1, w2, f1, f2, phi1=0.0, phi2=0.0):
    """Two-tone signal with a DC offset.

    S(t) = w0 + w1 · sin(2π f1 t + φ1) + w2 · sin(2π f2 t + φ2)

    Args:
        t (ndarray):  time samples in seconds, shape (n_samples,)
        w0 (float):   DC offset (0 Hz component)
        w1, w2 (float): amplitudes of the two sine components
        f1, f2 (float): frequencies in Hz
        phi1, phi2 (float): phase offsets in radians (default 0)

    Returns:
        ndarray: S(t), shape (n_samples,)
    """
    ####################################################
    ## TODO for students: implement the composite signal.
    # Fill out the function and remove the line below.
    #raise NotImplementedError("Student exercise: implement the composite signal S(t)")
    ####################################################

    # Compute S(t) = w0 + w1·sin(2π f1 t + φ1) + w2·sin(2π f2 t + φ2)
    s = (w0
         + w1 * np.sin(2 * np.pi * f1 * t + phi1)
         + w2 * np.sin(2 * np.pi * f2 * t + phi2))

    return s

[*Click for solution*](https://github.com/NeuromatchAcademy/course-content/tree/main/tutorials/W2D2_SignalProcessing/solutions/W2D2_Tutorial2_Solution_make_signal.py)

In [ ]:
# @title Solution

def make_signal(t, w0, w1, w2, f1, f2, phi1=0.0, phi2=0.0):
    s = (w0
         + w1 * np.sin(2 * np.pi * f1 * t + phi1)
         + w2 * np.sin(2 * np.pi * f2 * t + phi2))
    return s

In [14]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Exercise_2_Composite")

### Reading the demo

The demo below shows three views of the same signal:

- **Top — Reconstruction (the sum):** your reconstructed signal (blue) overlaid on the ground truth (gray dashed). This is the *sum* of the three weighted components — what the listener/observer actually "sees." The **MSE** badge in the corner reports how close your sum is to the truth.
- **Bottom-left — Components:** the three weighted pieces drawn *separately* before they are added. The dotted cyan line is the DC offset $w_0$ (a flat line at height $w_0$), and the orange and purple curves are $w_1\sin(2\pi f_1 t)$ and $w_2\sin(2\pi f_2 t)$. Adding these three curves point-by-point gives you the blue curve above.
- **Bottom-right — Magnitude Spectrum:** one bar per frequency, with bar height equal to $|w|$ for that frequency. The 0 Hz bar is the DC weight, and the bars at $f_1$ and $f_2$ are the amplitudes of the two sines. This is the *same* information as the bottom-left panel, but indexed by frequency instead of time — it's the picture a Fourier analysis would hand you.

As you move the sliders, watch all three update together: the spectrum bars change height, the component curves rescale, and the sum on top morphs toward the ground truth.

In [15]:
# @title Interactive Demo 2: Decompose the signal

def build_decomposition_demo():
    # ── Instructor config ──
    F1, F2 = 3, 7
    W0_TRUE, W1_TRUE, W2_TRUE = 0.8, 0.6, 1.4
    DURATION, FS = 1.0, 1000
    t = np.linspace(0, DURATION, int(FS * DURATION), endpoint=False)
    y_true = (W0_TRUE
              + W1_TRUE * np.sin(2 * np.pi * F1 * t)
              + W2_TRUE * np.sin(2 * np.pi * F2 * t))
    rng = np.random.default_rng()
    PHI1_RAND, PHI2_RAND = rng.uniform(0, 2 * np.pi, 2)

    # Harmonic structure for the spectrum panel (0 Hz prepended for the DC term)
    _F0_spec = math.gcd(int(F1), int(F2))
    _all_h   = [0] + list(range(1, int(max(F1, F2)) // _F0_spec + 1))
    _h1      = int(F1) // _F0_spec
    _h2      = int(F2) // _F0_spec

    # Current phases (interpolated during animation)
    _phases = [0.0, 0.0]

    img_out = widgets.Image(format='png')


    def draw(phases):
        phi1, phi2 = phases
        w0 = w0_slider.value
        w1 = w1_slider.value
        w2 = w2_slider.value

        c1        = w1 * np.sin(2 * np.pi * F1 * t + phi1)
        c2        = w2 * np.sin(2 * np.pi * F2 * t + phi2)
        y_student = make_signal(t, w0, w1, w2, F1, F2, phi1, phi2)
        mse       = np.mean((y_true - y_student) ** 2)

        # Phase-independent y-limits: use DC offset + sum of |coefficients| as upper
        # bound so the axis only moves when magnitudes change, not during animation
        y_abs_max = max(abs(W0_TRUE) + abs(W1_TRUE) + abs(W2_TRUE),
                        abs(w0) + abs(w1) + abs(w2), 0.5) * 1.2
        ylim      = (-y_abs_max, y_abs_max)

        spec_h, spec_c = [], []
        for h in _all_h:
            if h == 0:
                spec_h.append(abs(w0));  spec_c.append('#0891b2')
            elif h == _h1:
                spec_h.append(abs(w1));  spec_c.append('#e85d04')
            elif h == _h2:
                spec_h.append(abs(w2));  spec_c.append('#7c3aed')
            else:
                spec_h.append(0.0);      spec_c.append('#dddddd')
        spec_labels = [f'{h * _F0_spec} Hz' for h in _all_h]

        fig = plt.figure(figsize=(12, 9))
        gs      = fig.add_gridspec(2, 2)
        ax_main = fig.add_subplot(gs[0, :])
        ax_comp = fig.add_subplot(gs[1, 0])
        ax_spec = fig.add_subplot(gs[1, 1])

        # ── Top: reconstruction ───────────────────────────────
        ax_main.plot(t, y_true, color='#888888', linestyle='--', linewidth=2,
                     label='Ground truth', alpha=0.85)
        ax_main.plot(t, y_student, color='#2563EB', linewidth=2,
                     label='Your signal')
        ax_main.set_title('Reconstruction  (sum of components)', fontsize=13)
        ax_main.set_xlabel('Time (s)')
        ax_main.set_ylabel('Amplitude')
        ax_main.legend(loc='upper right')
        ax_main.set_xlim(0, DURATION)
        ax_main.set_ylim(ylim)
        ax_main.axhline(0, color='#cccccc', linewidth=0.8, zorder=0)

        mse_color = '#16a34a' if mse < 0.01 else '#dc2626'
        ax_main.text(0.02, 0.95, f'MSE: {mse:.4f}',
                     transform=ax_main.transAxes, fontsize=12, verticalalignment='top',
                     color=mse_color,
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                               edgecolor=mse_color, alpha=0.8))

        # ── Bottom-left: components overlaid ─────────────────
        phi1_txt = f' + φ={phi1:.2f}' if abs(phi1) > 1e-3 else ''
        phi2_txt = f' + φ={phi2:.2f}' if abs(phi2) > 1e-3 else ''
        ax_comp.plot(t, np.full_like(t, w0), color='#0891b2', linewidth=2,
                     linestyle=':', label=f'w₀={w0:.2f}  (DC)')
        ax_comp.plot(t, c1, color='#e85d04', linewidth=2,
                     label=f'w₁={w1:.2f}·sin(2π·{F1}Hz·t{phi1_txt})')
        ax_comp.plot(t, c2, color='#7c3aed', linewidth=2,
                     label=f'w₂={w2:.2f}·sin(2π·{F2}Hz·t{phi2_txt})')
        ax_comp.set_title('Components', fontsize=12)
        ax_comp.set_xlabel('Time (s)')
        ax_comp.set_ylabel('Amplitude')
        ax_comp.legend(loc='upper right', fontsize=9)
        ax_comp.set_xlim(0, DURATION)
        ax_comp.set_ylim(ylim)
        ax_comp.axhline(0, color='#cccccc', linewidth=0.8, zorder=0)

        # ── Bottom-right: magnitude spectrum ──────────────────
        ax_spec.bar(_all_h, spec_h, color=spec_c, width=0.55, alpha=0.9)
        ax_spec.set_xticks(_all_h)
        ax_spec.set_xticklabels(spec_labels, fontsize=9)
        ax_spec.set_title('Magnitude Spectrum', fontsize=12)
        ax_spec.set_ylabel('|Coefficient|')
        ax_spec.set_ylim(0, max(3.2, max(spec_h) * 1.3))
        ax_spec.axhline(0, color='#cccccc', linewidth=0.8, zorder=0)

        fig.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img_out.value = buf.read()


    def update(_change=None):
        draw(_phases)


    def on_phase_change(change):
        target = [PHI1_RAND, PHI2_RAND] if change['new'].startswith('Random') else [0.0, 0.0]
        n_frames, duration = 20, 1.6
        dt    = duration / n_frames
        start = list(_phases)
        for i in range(1, n_frames + 1):
            alpha      = i / n_frames
            _phases[0] = start[0] + alpha * (target[0] - start[0])
            _phases[1] = start[1] + alpha * (target[1] - start[1])
            draw(_phases)
            if i < n_frames:
                time.sleep(dt)


    w0_slider = widgets.FloatSlider(
        value=0.0, min=0.0, max=3.0, step=0.05,
        description='w₀  (DC, 0 Hz)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    w1_slider = widgets.FloatSlider(
        value=0.0, min=0.0, max=3.0, step=0.05,
        description=f'w₁  (f={F1} Hz)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    w2_slider = widgets.FloatSlider(
        value=0.0, min=0.0, max=3.0, step=0.05,
        description=f'w₂  (f={F2} Hz)',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    phase_radio = widgets.RadioButtons(
        options=['No phase  (φ = 0)', 'Random phase  (fixed per session)'],
        value='No phase  (φ = 0)',
        description='Phase:',
        style={'description_width': 'initial'}
    )

    w0_slider.observe(update, names='value')
    w1_slider.observe(update, names='value')
    w2_slider.observe(update, names='value')
    phase_radio.observe(on_phase_change, names='value')

    ui = widgets.VBox([
        widgets.HBox([w0_slider, w1_slider, w2_slider]),
        phase_radio,
        img_out
    ])

    display(ui)
    update()


build_decomposition_demo()

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Demo_2_Decomposition")

## Think! 2: Phase and the spectrum

Once you are able to match your signal to the ground truth mystery signal, try toggling between the "random phase" and "no phase" radio buttons. This changes *only* the phases of the component signals without changing their amplitudes.

- What do you observe happens to the reconstructed signal in the top panel?
- What happens to the magnitude spectrum (bottom-right) when only the phases change?

In [16]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Think_2_Phase")

---

# Section 3: The Fourier Series

*Estimated timing to here from start of tutorial: min*


## Video 3: Building complex signals from sine waves

In [17]:
from IPython.display import HTML
HTML("<div style='background:#eee;padding:1em;border-radius:6px;'>"
     "<b>Video 3 placeholder.</b> Topics: harmonics, Fourier coefficients, convergence, and the Gibbs phenomenon."
     "</div>")

In [18]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Video_3_FourierSeries")

Familiar periodic signals — square waves and sawtooth waves — look nothing like sine waves. Yet every one of them can be written as a (theoretically infinite) sum of sine waves:

$$x(t) = a_1 \sin(2\pi f_0 t) + a_2 \sin(2\pi f_2 t) + a_3 \sin(2\pi f_3 t) + \cdots$$

where the frequencies $f_1, f_2, \ldots$ are **fixed harmonics** of a fundamental frequency $f_0 = 2$ Hz, and the $a_n$ are the **Fourier coefficients** that determine how much of each frequency is present.

Here we look at the **first 5 components** — enough to recognise the shape of each signal.

**Your task:** choose the coefficient for each component using the sliders and try to match the target signal (gray dashed) — the *actual* square / sawtooth wave. With only 5 terms you **cannot** drive the **MSE** all the way to 0; instead, aim for the **best possible 5-term fit**. When your coefficients reach it the badge turns **green**, and the MSE it shows is the leftover **truncation error** — the gap between 5 sine waves and the true wave. (You'll close that gap by adding more terms in the next section.)

> **Hint:** not all coefficients are positive — some components are subtracted rather than added. Look at what happens to the shape in the bottom panel as you go negative.
>
> A **negative coefficient** is equivalent to a positive one with a **phase shift of $\pi$** — the component is simply flipped upside-down. So the sign of a slider tells you the phase: positive means $\phi = 0$, negative means $\phi = \pi$. In the magnitude spectrum the bar height is always the **positive magnitude** $|a_n|$; the sign/phase is shown by the hatch.

## Interactive Demo 3: Building signals from sine waves

In [19]:
# @title Interactive Demo 3: Building signals from sine waves

def build_fourier_builder_demo():
    # ── Instructor config ──
    F0 = 2.0
    DURATION, FS = 1.0, 2000
    t = np.linspace(0, DURATION, int(FS * DURATION), endpoint=False)
    rng = np.random.default_rng()
    RAND_PHASES = rng.uniform(0, 2 * np.pi, 5)

    COMP_COLORS = ['#e85d04', '#7c3aed', '#059669', '#d97706', '#0891b2']

    SIGNAL_DEFS = {
        'Square wave': {
            'harmonics': [1, 3, 5, 7, 9],
            'coeffs':    [4 / (n * np.pi) for n in [1, 3, 5, 7, 9]],
        },
        'Sawtooth wave': {
            'harmonics': [1, 2, 3, 4, 5],
            'coeffs':    [2 * (-1)**(n + 1) / (n * np.pi) for n in [1, 2, 3, 4, 5]],
        },
    }

    def true_signal(signal_type):
        """Analytic ground-truth waveform (amplitude ±1) — the *actual* wave,
        not a Fourier sum. This is what the student's reconstruction is scored
        against, so the displayed MSE never quite reaches 0 with only 5 terms."""
        x = 2 * np.pi * F0 * t
        if signal_type == 'Square wave':
            return np.where(np.sin(x) >= 0, 1.0, -1.0)
        else:  # Sawtooth wave
            return 2 * ((F0 * t + 0.5) % 1.0) - 1.0

    _phases = [0.0] * 5

    img_out = widgets.Image(format='png')


    def build_signal(coeffs, freqs, phases):
        return sum(a * np.sin(2 * np.pi * f * t + phi)
                   for a, f, phi in zip(coeffs, freqs, phases))


    def draw(phases):
        signal_type    = signal_dropdown.value
        defn           = SIGNAL_DEFS[signal_type]
        harmonics      = defn['harmonics']
        freqs          = [n * F0 for n in harmonics]
        true_coeffs    = defn['coeffs']
        student_coeffs = [s.value for s in sliders]

        for slider, n, f in zip(sliders, harmonics, freqs):
            slider.description = f'a{n}  ({f:.0f} Hz)'

        # y_target: the actual analytic wave (gray dashed) — what we score against.
        # y_best:   the best achievable 5-term Fourier fit (true coefficients,
        #           zero phase). Matching it is the goal, and it drives the
        #           green/red trigger exactly as before.
        y_target   = true_signal(signal_type)
        y_best     = build_signal(true_coeffs, freqs, [0.0] * 5)
        y_student  = build_signal(student_coeffs, freqs, phases)
        components = [a * np.sin(2 * np.pi * f * t + phi)
                      for a, f, phi in zip(student_coeffs, freqs, phases)]

        # Displayed MSE: distance to the *actual* wave (floors at the truncation
        # residual, so it stays >0 even for a perfect 5-term fit).
        mse_display = np.mean((y_target - y_student) ** 2)
        # Trigger MSE: distance to the best 5-term fit — "the way it is now".
        mse_trigger = np.mean((y_best - y_student) ** 2)
        is_matched  = mse_trigger < 0.01

        # Phase-independent y-limits: use sum of |coefficients| as upper bound
        # so the axis only moves when magnitudes change, not during phase animation
        y_abs_max = max(
            sum(abs(c) for c in true_coeffs),
            sum(abs(c) for c in student_coeffs),
            float(np.max(np.abs(y_target))),
            0.5
        ) * 1.2
        ylim = (-y_abs_max, y_abs_max)

        # Magnitude spectrum data
        max_h   = max(harmonics)
        all_h   = list(range(1, max_h + 1))
        active  = dict(zip(harmonics, range(len(harmonics))))

        spec_heights = []
        spec_colors  = []
        spec_hatches = []
        spec_labels  = [f'{h*F0:.0f} Hz' for h in all_h]

        for h in all_h:
            if h in active:
                coeff = student_coeffs[active[h]]
                spec_heights.append(abs(coeff))
                spec_colors.append(COMP_COLORS[active[h]])
                spec_hatches.append('///' if coeff < 0 else '')
            else:
                spec_heights.append(0.0)
                spec_colors.append('#dddddd')
                spec_hatches.append('')

        fig = plt.figure(figsize=(12, 9))
        gs      = fig.add_gridspec(2, 2)
        ax_main = fig.add_subplot(gs[0, :])
        ax_comp = fig.add_subplot(gs[1, 0])
        ax_spec = fig.add_subplot(gs[1, 1])

        # ── Top: reconstruction vs the true wave ──────────────
        ax_main.plot(t, y_target, color='#888888', linestyle='--', linewidth=2,
                     label=f'Target (true {signal_type.lower()})', alpha=0.85)
        ax_main.plot(t, y_student, color='#2563EB', linewidth=2,
                     label='Your reconstruction')
        ax_main.set_title(f'{signal_type} — Reconstruction', fontsize=13)
        ax_main.set_xlabel('Time (s)')
        ax_main.set_ylabel('Amplitude')
        ax_main.legend(loc='upper right')
        ax_main.set_xlim(0, DURATION)
        ax_main.set_ylim(ylim)
        ax_main.axhline(0, color='#cccccc', linewidth=0.8, zorder=0)

        mse_color = '#16a34a' if is_matched else '#dc2626'
        ax_main.text(0.02, 0.95, f'MSE: {mse_display:.4f}',
                     transform=ax_main.transAxes, fontsize=12, verticalalignment='top',
                     color=mse_color,
                     bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                               edgecolor=mse_color, alpha=0.8))

        # ── Bottom-left: components overlaid ─────────────────
        phi_mode = phase_radio.value.startswith('Random')
        for comp, f, coeff, phi, n in zip(components, freqs, student_coeffs, phases, harmonics):
            phi_lbl = f'+φ{n}' if phi_mode and abs(phi) > 1e-3 else ''
            ax_comp.plot(t, comp, color=COMP_COLORS[active[n]], linewidth=1.8,
                         label=f'a{n}={coeff:.2f}·sin(2π·{f:.0f}Hz·t{phi_lbl})')
        ax_comp.set_title('Components', fontsize=12)
        ax_comp.set_xlabel('Time (s)')
        ax_comp.set_ylabel('Amplitude')
        ax_comp.legend(loc='upper right', fontsize=8)
        ax_comp.set_xlim(0, DURATION)
        ax_comp.set_ylim(ylim)
        ax_comp.axhline(0, color='#cccccc', linewidth=0.8, zorder=0)

        # ── Bottom-right: magnitude spectrum ──────────────────
        for h, height, color, hatch in zip(all_h, spec_heights, spec_colors, spec_hatches):
            ax_spec.bar(h, height, color=color, width=0.55, alpha=0.85,
                        hatch=hatch, edgecolor='#444444' if hatch else color,
                        linewidth=0.8)

        legend_handles = [
            Patch(facecolor='#aaaaaa', edgecolor='#444444', label='positive amplitude'),
            Patch(facecolor='#aaaaaa', edgecolor='#444444', hatch='///',
                  label='negative amplitude (flipped phase)'),
        ]
        ax_spec.legend(handles=legend_handles, fontsize=8, loc='upper right')

        ax_spec.set_xticks(all_h)
        ax_spec.set_xticklabels(spec_labels, fontsize=8)
        ax_spec.set_title('Magnitude Spectrum', fontsize=12)
        ax_spec.set_ylabel('|Coefficient|')
        ax_spec.set_ylim(0, max(2.2, max(spec_heights) * 1.3))
        ax_spec.axhline(0, color='#cccccc', linewidth=0.8, zorder=0)

        fig.tight_layout()
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img_out.value = buf.read()


    def update(_change=None):
        draw(_phases)


    def on_phase_change(change):
        target = list(RAND_PHASES[:5]) if change['new'].startswith('Random') else [0.0] * 5
        n_frames, duration = 20, 1.6
        dt    = duration / n_frames
        start = list(_phases)
        for i in range(1, n_frames + 1):
            alpha = i / n_frames
            for j in range(5):
                _phases[j] = start[j] + alpha * (target[j] - start[j])
            draw(_phases)
            if i < n_frames:
                time.sleep(dt)


    def on_signal_change(change):
        for slider in sliders:
            slider.unobserve(update, names='value')
        for slider in sliders:
            slider.value = 0.0
        for slider in sliders:
            slider.observe(update, names='value')
        update()


    def on_reset(_btn):
        for slider in sliders:
            slider.unobserve(update, names='value')
        for slider in sliders:
            slider.value = 0.0
        for slider in sliders:
            slider.observe(update, names='value')
        update()


    signal_dropdown = widgets.Dropdown(
        options=list(SIGNAL_DEFS.keys()),
        value='Square wave',
        description='Signal type:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='280px')
    )
    sliders = [
        widgets.FloatSlider(
            value=0.0, min=-2.0, max=2.0, step=0.01,
            description='',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='550px'),
            readout_format='.2f'
        )
        for i in range(5)
    ]
    phase_radio = widgets.RadioButtons(
        options=['No phase  (φ = 0)', 'Random phase  (fixed per session)'],
        value='No phase  (φ = 0)',
        description='Phase:',
        style={'description_width': 'initial'}
    )
    reset_btn = widgets.Button(
        description='Reset coefficients',
        button_style='warning',
        layout=widgets.Layout(width='180px')
    )

    signal_dropdown.observe(on_signal_change, names='value')
    phase_radio.observe(on_phase_change, names='value')
    reset_btn.on_click(on_reset)
    for slider in sliders:
        slider.observe(update, names='value')

    note = widgets.HTML(
        value="<i>A <b>negative</b> coefficient flips its component upside-down — "
              "it is the same as a phase shift of π.</i>"
    )

    ui = widgets.VBox([
        widgets.HBox([signal_dropdown, reset_btn]),
        phase_radio,
        note,
        widgets.VBox(sliders),
        img_out
    ])

    display(ui)
    update()


build_fourier_builder_demo()

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Demo_3_FourierBuilder")

## Think! 3: Look at the spectra

Before moving on, cycle through the three signal types in the demo above and pay attention to the **magnitude spectrum** (bottom-right) and the **signs of the slider values** you needed to match each target. A few things to reflect on:

**1. What do you observe about the *component frequencies* in each wave?**
Which harmonics actually appear? Are all of them used, or only some? Compare the square and sawtooth waves — you should see that one of them uses only *odd* harmonics (1·f₀, 3·f₀, 5·f₀, …) while the other uses *every* harmonic (1·f₀, 2·f₀, 3·f₀, …).

**2. What do you observe about the *pattern of signs* on the weights?**
Are all the coefficients positive, or do some need to be negative? Is there a regular pattern (e.g. + + + +, or + − + −, or all positive)? Remember that a negative coefficient is *not* a different kind of component — it is just the same sine wave with a **phase of π** (i.e. flipped upside-down). So the sign pattern is really a phase pattern.

**3. What do you observe about the *magnitudes* of the weights?**
Are the weights all the same size, or do they shrink as the frequency increases? If they shrink, do they decrease *linearly* (like 1, 1/2, 1/3, 1/4, …) or *faster* (like 1, 1/9, 1/25, …)? This rate of decay is what determines how quickly the Fourier series converges — and it's the reason some waveforms need many more harmonics than others to look "right."

Keep your answers in mind as you move into the next section, where we add more and more components and watch the approximation improve.

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Think_3_Spectra")

## How Many Components Does It Take?

So far you've been working with just the first **5** Fourier components. But in theory these signals require *infinitely many* — so how much does adding more actually help?

Use the slider below to increase **N**, the number of sine-wave components included in the approximation. The coefficients are set automatically to the correct Fourier values — you're just controlling how many terms to keep. The approximation (blue) is compared against the **true signal** (gray dashed) — the actual square, sawtooth, or Gaussian pulse, not another Fourier sum.

A few things to watch for as you increase N:
- **Sawtooth wave** converges slowly — its coefficients shrink only as 1/n.
- **Square wave** is the most stubborn: you'll notice the "ringing" overshoot near each edge (**Gibbs phenomenon**) that never fully disappears no matter how many terms you add — it just gets narrower.
- **Gaussian pulse** is something new — an **aperiodic** signal: a single localized bump rather than a repeating wave. Yet it too is *just a sum of plain sine waves*. A sharp, narrow pulse needs **many** harmonics (a broad band of frequencies) to build up; with only a few terms it stays a low, wide blob.

### From magnitude to power

Up to now the bottom panel has shown a **magnitude** spectrum — the size $|a_n|$ of each component. Here we switch to the **power spectrum**: the power carried by each harmonic, $P_n = |a_n|^2/2$ (the average power of a sine of amplitude $a_n$). Power is what most measurements — and our ears — actually respond to, and squaring exaggerates the tall components and flattens the small ones, so the *shape* of the spectrum stands out. We held this back in the earlier demos to keep the focus on building and reading signals; here it earns its place.

**Why introduce it here?** Because the power spectrum is secretly what the convergence badge has been measuring all along. By **Parseval's theorem** the total power of a signal equals the sum of its harmonic powers — so the **MSE between your N-term sum and the true signal is exactly the total power of the harmonics you left out**. The bars you drop *are* the error you see; watch the MSE fall as more bars switch on.

**Reading the envelope.** The harmonics are drawn as stems sampling a smooth gray **envelope** — the transform of *one period* of the signal. Its shape says a lot:
- **Square wave** → a **sinc²** envelope: a fat main lobe, then **nulls** (every even harmonic lands exactly in a null, with zero power) and decaying **side lobes** carrying the odd harmonics. Hard edges in time cost you a long, slowly-decaying tail of frequencies. (You'll meet this same sinc again when we get to filtering.)
- **Sawtooth wave** → a smooth **1/f²** roll-off, no nulls — every harmonic present, power shrinking steadily.
- **Gaussian pulse** → a **Gaussian** envelope: smooth, no nulls, falling off fastest of all. A smooth bump needs the *fewest* high frequencies; a sharp edge needs the most.

Use the **Power axis** toggle to switch how the vertical scale is drawn:
- **dB** (decibels — log power against linear frequency): stretches the small values so the square's nulls and side lobes, and the long high-frequency tails, stand out clearly.
- **Log-log** (log power against log frequency): the natural view for **power laws**. A spectrum that falls as $1/f^{\alpha}$ becomes a **straight line of slope $-\alpha$** — the sawtooth is a clean slope $-2$, and the square's side-lobe peaks ride a slope $-2$ line with deep notches between them. The Gaussian, which decays faster than *any* power law, instead **curves over and drops off a cliff**. This is exactly how aperiodic, 1/f-like neural power spectra are read.

**The width slider** (Gaussian only) makes the time–frequency trade-off tangible: **widen** σ and the pulse spreads out in time while its spectrum **narrows** — a broad, smooth bump is built from just a few low frequencies; **shrink** σ and the pulse sharpens while its spectrum **broadens** toward higher frequencies. Time-width and frequency-width move in *opposite* directions ($\Delta t \cdot \Delta f \approx$ constant) — the same reciprocity behind the uncertainty principle. Watch the converged pulse, too: a wide pulse needs only a handful of harmonics, a narrow one needs many.

### Same spectrum, very different signal: phase

Select **Gaussian pulse**, then toggle **Aligned phase → pulse** against **Random phase → noise**. This changes *only* the phases of the components, and it **animates**: you'll watch the sharp pulse smoothly melt into noise (and morph back) while the **power spectrum stays frozen** — the bottom panel never moves. Yet the signal completely changes:

- **Aligned:** all the sine waves crest at the *same instant*, add up constructively, and cancel elsewhere → a single **sharp pulse**.
- **Random:** the *same* amplitudes now crest at scattered times, so the energy smears across the whole window → **noise**.

The power spectrum alone does **not** determine a signal. It tells you *which frequencies and how much* — but it's the **phase** that says *when* things happen. A spike and a burst of noise can have an identical spectrum and differ only in their phases.

In [20]:
# @title Interactive Demo 4: Fourier series convergence

def build_convergence_demo():
    # ── Instructor config ──
    F0 = 2.0                        # fundamental of the square / sawtooth (Hz)
    DURATION, FS = 1.0, 2000
    t = np.linspace(0, DURATION, int(FS * DURATION), endpoint=False)

    GAUSS_F0 = 1.0 / DURATION       # one pulse per window → a single localised bump

    # Fixed random phases so that raising N just adds more scrambled terms.
    _rng        = np.random.default_rng(0)
    RAND_PHASES = _rng.uniform(0, 2 * np.pi, 256)

    # Current Gaussian phases (interpolated during the pulse ↔ noise animation)
    _phases = []

    def gauss_sigma():
        """Gaussian pulse width in seconds, read live from the width slider."""
        return gauss_width_slider.value / 1000.0


    def true_signal(signal_type):
        """Analytic ground-truth waveform."""
        x = 2 * np.pi * F0 * t
        if signal_type == 'Square wave':
            return np.where(np.sin(x) >= 0, 1.0, -1.0)
        elif signal_type == 'Sawtooth wave':
            return 2 * ((F0 * t + 0.5) % 1.0) - 1.0
        else:  # Gaussian pulse — a single narrow, centred bump (an aperiodic transient)
            sigma = gauss_sigma()
            return np.exp(-((t - DURATION / 2) ** 2) / (2 * sigma ** 2))


    def spectrum(signal_type, N):
        """First-N-harmonic description of the signal.

        Every signal is written in the single form
            x(t) = DC + Σ |aₙ|·sin(2π fₙ t + φₙ),
        so the phases can later be swapped for random values *without touching
        the magnitudes* — the heart of the Gaussian phase demo. Returns the
        present harmonics, their frequencies, magnitudes |aₙ|, the aligned
        phases, the DC term, the fundamental, and the full harmonic grid (so the
        square's missing even harmonics can be marked as zeros).
        """
        if signal_type == 'Square wave':
            harm   = [2 * k + 1 for k in range(N)]
            mags   = [4 / (h * np.pi) for h in harm]
            phases = [0.0 for _ in harm]
            dc, fundamental = 0.0, F0
            grid   = list(range(1, harm[-1] + 1))
        elif signal_type == 'Sawtooth wave':
            harm   = list(range(1, N + 1))
            mags   = [2 / (h * np.pi) for h in harm]
            phases = [0.0 if h % 2 == 1 else np.pi for h in harm]   # sign (-1)^(h+1)
            dc, fundamental = 0.0, F0
            grid   = list(harm)
        else:  # Gaussian pulse — true coefficients straight from the FFT
            g          = true_signal('Gaussian pulse')
            coeffs     = np.fft.rfft(g) / len(g)
            harm       = list(range(1, N + 1))
            mags       = [2 * np.abs(coeffs[h]) for h in harm]
            phases     = [np.angle(coeffs[h]) + np.pi / 2 for h in harm]  # cos → sin
            dc, fundamental = float(coeffs[0].real), GAUSS_F0
            grid       = list(harm)
        freqs = [h * fundamental for h in harm]
        return harm, freqs, mags, phases, dc, fundamental, grid


    def envelope_power(signal_type, nu):
        """Continuous power envelope |X(ν)|²/2 that the harmonics sample, as a
        function of (continuous) harmonic number ν. This is the transform of one
        period: a sinc² for the square (boxcar edges → nulls + side lobes), a
        smooth 1/ν² roll-off for the sawtooth ramp, and a Gaussian for the
        Gaussian pulse. The discrete harmonic powers land exactly on it."""
        if signal_type == 'Square wave':
            return 2.0 * np.sinc(nu / 2.0) ** 2          # np.sinc(x)=sin(πx)/(πx)
        elif signal_type == 'Sawtooth wave':
            return 2.0 / (np.pi ** 2 * nu ** 2)
        else:  # Gaussian pulse — width set by the slider
            sigma = gauss_sigma()
            return 4 * np.pi * sigma ** 2 * np.exp(-4 * np.pi ** 2 * sigma ** 2 * nu ** 2)


    def reconstruct(freqs, mags, phases, dc):
        y = np.full_like(t, dc)
        for f, m, p in zip(freqs, mags, phases):
            y = y + m * np.sin(2 * np.pi * f * t + p)
        return y


    def target_phases(N):
        """The Gaussian phase vector for the *current* radio mode at this N."""
        _, _, _, aligned, _, _, _ = spectrum('Gaussian pulse', N)
        if phase_radio.value.startswith('Random'):
            return list(RAND_PHASES[:N])
        return list(aligned)


    conv_dropdown = widgets.Dropdown(
        options=['Square wave', 'Sawtooth wave', 'Gaussian pulse'],
        value='Square wave',
        description='Signal type:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='280px')
    )

    n_slider = widgets.IntSlider(
        value=1, min=1, max=200, step=1,
        description='N components:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )

    phase_radio = widgets.RadioButtons(
        options=['Aligned phase  → pulse', 'Random phase  → noise'],
        value='Aligned phase  → pulse',
        description='Gaussian phase:',
        style={'description_width': 'initial'},
        disabled=True
    )

    scale_toggle = widgets.ToggleButtons(
        options=['dB', 'Log-log'],
        value='dB',
        description='Power axis:',
        style={'description_width': 'initial'}
    )

    gauss_width_slider = widgets.IntSlider(
        value=8, min=4, max=30, step=1,
        description='Gaussian width σ (ms):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px'),
        disabled=True
    )

    conv_img = widgets.Image(format='png')


    def draw(gauss_phases):
        signal_type = conv_dropdown.value
        N           = n_slider.value
        is_gauss    = signal_type == 'Gaussian pulse'

        harm, freqs, mags, phases, dc, fundamental, grid = spectrum(signal_type, N)
        recon_phases = gauss_phases if is_gauss else phases
        randomized   = is_gauss and phase_radio.value.startswith('Random')

        y_ref    = true_signal(signal_type)
        y_approx = reconstruct(freqs, mags, recon_phases, dc)
        mse      = np.mean((y_ref - y_approx) ** 2)

        max_h = max(grid)

        fig = plt.figure(figsize=(10, 7.5), constrained_layout=True)
        gs  = fig.add_gridspec(2, 1, height_ratios=[3, 2])
        ax      = fig.add_subplot(gs[0])
        ax_spec = fig.add_subplot(gs[1])

        # ── Top: reconstruction vs the true signal ──
        approx_lbl = (f'{N}-term sum (phases scrambled)' if randomized
                      else f'{N}-term approximation')
        ax.plot(t, y_ref, color='#888888', linestyle='--', linewidth=2,
                label=f'Ground truth {signal_type.lower()}', alpha=0.85)
        ax.plot(t, y_approx, color='#2563EB', linewidth=2, label=approx_lbl)
        gauss_tail = f'  —  σ = {gauss_width_slider.value} ms' if is_gauss else ''
        title_tail = '  —  phases scrambled → noise' if randomized else gauss_tail
        ax.set_title(f'{signal_type} — Fourier Series  (N = {N}){title_tail}', fontsize=13)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Amplitude')
        ax.legend(loc='upper right')
        ax.set_xlim(0, DURATION)
        ax.set_ylim((-0.6, 1.25) if is_gauss else (-1.35, 1.35))
        ax.axhline(0, color='#cccccc', linewidth=0.8, zorder=0)

        mse_color = '#16a34a' if mse < 0.005 else '#dc2626'
        ax.text(0.02, 0.95, f'MSE vs ground truth: {mse:.5f}',
                transform=ax.transAxes, fontsize=11, verticalalignment='top',
                color=mse_color,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor=mse_color, alpha=0.8))

        # ── Bottom: POWER spectrum — discrete harmonics sampling a continuous
        #            envelope (the transform of one period) ──
        powers       = [m ** 2 / 2 for m in mags]            # Pₙ = |aₙ|²/2
        missing_h    = [h for h in grid if h not in set(harm)]
        missing_freq = [h * fundamental for h in missing_h]

        nu       = np.linspace(0.5, max_h + 1, 800)
        env      = envelope_power(signal_type, nu)
        env_freq = nu * fundamental
        pmax     = max(powers) if powers else 1.0

        mode = scale_toggle.value
        if mode == 'dB':
            FLOOR = -40.0
            to_db = lambda P: np.clip(10 * np.log10(np.maximum(np.asarray(P, float), 1e-12) / pmax),
                                      FLOOR, None)
            env_y     = to_db(env)
            present_y = to_db(powers)
            missing_y = np.full(len(missing_freq), FLOOR)
            base      = FLOOR
            ylabel    = 'Power  (dB)'
            xlim      = (0, (max_h + 1) * fundamental)
            ylim      = (FLOOR, 5)
        else:  # Log-log
            floor     = pmax * 1e-5
            env_y     = np.maximum(env, floor)
            present_y = np.maximum(np.asarray(powers, float), floor)
            missing_y = np.full(len(missing_freq), floor)
            base      = floor
            ylabel    = 'Power  (log)'
            xlim      = (min(freqs) * 0.8, (max_h + 1) * fundamental)
            ylim      = (floor, pmax * 2)

        ax_spec.plot(env_freq, env_y, color='#9aa0a6', linewidth=1.6, zorder=1,
                     label='envelope (transform of one period)')
        ax_spec.vlines(freqs, base, present_y, color='#2563EB', linewidth=1.2,
                       alpha=0.55, zorder=2)
        ax_spec.scatter(freqs, present_y, color='#2563EB', s=16, zorder=3,
                        label='harmonic')
        if missing_freq:
            ax_spec.scatter(missing_freq, missing_y, marker='x', color='#888888',
                            s=26, linewidths=1.2, zorder=3,
                            label='zero power (between harmonics)')

        spec_note = '  (identical for pulse and noise)' if is_gauss else ''
        ax_spec.set_title('Power Spectrum of the Components' + spec_note, fontsize=12)
        ax_spec.set_xlabel('Frequency (Hz)')
        ax_spec.set_ylabel(ylabel)
        if mode == 'Log-log':
            ax_spec.set_xscale('log')
            ax_spec.set_yscale('log')
        ax_spec.set_xlim(xlim)
        ax_spec.set_ylim(ylim)
        ax_spec.axhline(base, color='#cccccc', linewidth=0.8, zorder=0)
        ax_spec.legend(loc='upper right', fontsize=8)

        if mode == 'dB':
            # Thinned frequency ticks so large N stays readable (linear-freq view)
            step   = max(1, len(grid) // 10)
            tick_h = grid[::step]
            ax_spec.set_xticks([h * fundamental for h in tick_h])
            ax_spec.set_xticklabels([f'{h*fundamental:.0f}' for h in tick_h], fontsize=8)

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        conv_img.value = buf.read()


    def update_conv(_change=None):
        # Static redraw (slider / dropdown / scale / width). Refresh the Gaussian
        # phases to match the current radio mode at this N; others ignore them.
        if conv_dropdown.value == 'Gaussian pulse':
            _phases[:] = target_phases(n_slider.value)
        draw(_phases)


    def on_phase_change(change):
        # Animate the Gaussian smoothly between aligned-phase pulse and
        # random-phase noise (both directions). The power spectrum never changes
        # — only the phases morph.
        if conv_dropdown.value != 'Gaussian pulse':
            return
        N      = n_slider.value
        target = target_phases(N)                          # destination (new mode)
        start  = list(_phases) if len(_phases) == N else target_phases(N)

        n_frames, duration = 20, 1.6
        dt = duration / n_frames
        for i in range(1, n_frames + 1):
            a = i / n_frames
            _phases[:] = [s + a * (tt - s) for s, tt in zip(start, target)]
            draw(_phases)
            if i < n_frames:
                time.sleep(dt)


    def on_signal_change(change):
        # The phase + width controls only make sense for the Gaussian pulse
        is_gauss = (change['new'] == 'Gaussian pulse')
        phase_radio.disabled        = not is_gauss
        gauss_width_slider.disabled = not is_gauss
        update_conv()


    conv_dropdown.observe(on_signal_change, names='value')
    n_slider.observe(update_conv, names='value')
    phase_radio.observe(on_phase_change, names='value')
    scale_toggle.observe(update_conv, names='value')
    gauss_width_slider.observe(update_conv, names='value')

    display(widgets.VBox([
        widgets.HBox([conv_dropdown, n_slider]),
        widgets.HBox([phase_radio, scale_toggle]),
        gauss_width_slider,
        conv_img
    ]))
    update_conv()


build_convergence_demo()

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Demo_4_Convergence")

---

# Section 4: Visualizing power spectra of real data

The most common method for performing spectral analysis is the **Fast Fourier Transform (FFT)**. The FFT is an efficient algorithm to compute the Discrete Fourier Transform (DFT), which decomposes a signal from its original domain (often time) into a representation in the frequency domain.

Here's a basic overview of the process:

1.  **Time Domain Signal**: We start with a signal recorded over time (e.g., `raw_ap` in our notebook).
2.  **Fast Fourier Transform (FFT)**: We apply the FFT to this time-domain signal. The output of the FFT is a complex-valued array, where each element corresponds to a specific frequency. It essentially tells us the amplitude and phase of each frequency component present in the original signal.
3.  **Frequency Axis (`xf`)**: The `fftfreq` function (from `scipy.fft`) is used to generate the corresponding frequencies for each point in the FFT output.
4.  **Spectrum Calculation**: From the FFT output, we can derive different types of spectra:
    *   **Amplitude Spectrum**: This shows the magnitude (amplitude) of each frequency component. It's calculated as `2.0/N * np.abs(yf[0:N//2])`, where `N` is the number of samples and `yf` is the FFT output. The `2.0/N` scaling normalizes the amplitude, and `[0:N//2]` considers only the positive frequency components (the FFT output is symmetric).
    *   **Power Spectrum**: This shows the power of each frequency component. It's often calculated as the square of the amplitude spectrum, sometimes with an additional normalization (`(1.0/N * np.abs(yf[0:N//2]))**2`). Power spectra are useful because power is additive and directly related to the energy content at each frequency. This is often preferred when comparing the 'strength' of different frequency bands.

By plotting these spectra, we can visually identify dominant frequencies, broadband noise, or other spectral features in the signal.

In this section, we will look at some real data collected at the Internation Brain Laboratory using Neuropixels

In [33]:
# @title Submit your feedback on this tutorial
content_review(f"{feedback_prefix}_Tutorial")

In [21]:
# @title Install
!pip install remfile h5py --quiet

In [22]:
# @title Imports
import ipywidgets as widgets
from IPython.display import display

import remfile, h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq


## Load IBL Neuropixels Data — Streams directly from DANDI (run me first!)
Streams 2 seconds from subject DY-008, channel 60 (best LFP+spike channel). No large download needed — remfile fetches only the bytes we ask for.

In [23]:
url  = 'https://api.dandiarchive.org/api/assets/604b7457-430e-4160-b7ff-594562d942e3/download/'
f    = h5py.File(remfile.File(url), 'r')
ap   = f['acquisition']['ElectricalSeriesProbe00AP']

In [24]:
ap_fs  = 30000.0       # Hz, confirmed from timestamps
conv   = 2.34375e-06   # to volts
BEST_CH = 60           # channel with clearest LFP structure + spikes
T_DUR   = 2.0          # seconds to stream
i1      = int(T_DUR * ap_fs)
t_ap    = np.arange(i1) / ap_fs

In [25]:
print(f'Streaming {T_DUR}s from channel {BEST_CH} at {ap_fs} Hz...')

# Contiguous slice required by remfile — [:, CH:CH+1] not [:, CH]
raw_ap  = ap['data'][:i1, BEST_CH:BEST_CH+1].astype(float)[:, 0] * conv * 1e6  # µV
raw_ap  = raw_ap - np.mean(raw_ap)  # zero-centre

print(f'Done. Shape: {raw_ap.shape}, duration: {t_ap[-1]:.2f}s')
print(f'Signal range: {raw_ap.min():.1f} to {raw_ap.max():.1f} µV')

Streaming 2.0s from channel 60 at 30000.0 Hz...
Done. Shape: (60000,), duration: 2.00s
Signal range: -344.4 to 185.3 µV


In [31]:
# Perform FFT on the raw signal
N = len(raw_ap) # Number of sample points
T = 1.0 / ap_fs # Sample spacing (1/sampling frequency)

yf = fft(raw_ap)
xf = fftfreq(N, T)[:N//2]

# Create widgets for user control
spectrum_type_widget = widgets.RadioButtons(
    options=['Amplitude Spectrum', 'Power Spectrum'],
    value='Amplitude Spectrum',
    description='Spectrum Type:',
    disabled=False,
    layout=widgets.Layout(width='auto')
)

x_scale_widget = widgets.RadioButtons(
    options=['linear', 'log'],
    value='linear',
    description='X-axis Scale:',
    disabled=False,
    layout=widgets.Layout(width='auto')
)

y_scale_widget = widgets.RadioButtons(
    options=['linear', 'log'],
    value='linear',
    description='Y-axis Scale:',
    disabled=False,
    layout=widgets.Layout(width='auto')
)

output_img = widgets.Image(format='png')

def update_spectrum_plot(_change=None):
    spectrum_type = spectrum_type_widget.value
    x_scale = x_scale_widget.value
    y_scale = y_scale_widget.value

    fig, axes = plt.subplots(2, 1, figsize=(13, 10), sharex=False)

    # Top Panel: Raw Signal Over Time
    axes[0].plot(t_ap, raw_ap, color='C4', lw=0.6)
    axes[0].set_title('Raw Broadband Signal Over Time')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Amplitude (µV)')
    axes[0].grid(True, linestyle='--', alpha=0.7)

    # Bottom Panel: Spectrum Plot
    plot_xf = xf
    if spectrum_type == 'Amplitude Spectrum':
        plot_y_data = 2.0/N * np.abs(yf[0:N//2])
        y_label = 'Amplitude (µV)'
    else: # Power Spectrum
        plot_y_data = (1.0/N * np.abs(yf[0:N//2]))**2
        y_label = 'Power (µV$^2$)'

    # Exclude DC component (0 Hz) if x-axis is logarithmic to avoid issues with log(0)
    if x_scale == 'log':
        first_nonzero_freq_idx = np.where(plot_xf > 0)[0]
        if len(first_nonzero_freq_idx) > 0:
            start_idx = first_nonzero_freq_idx[0]
            axes[1].plot(plot_xf[start_idx:], plot_y_data[start_idx:], color='C1')
        else:
            axes[1].plot([1e-6], [0], color='C1')
    else:
        axes[1].plot(plot_xf, plot_y_data, color='C1')

    axes[1].set_title(f'{spectrum_type} of Raw Broadband Signal')
    axes[1].set_xlabel('Frequency (Hz)')
    axes[1].set_ylabel(y_label)
    axes[1].set_xscale(x_scale)
    axes[1].set_yscale(y_scale)
    axes[1].grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    output_img.value = buf.read()

spectrum_type_widget.observe(update_spectrum_plot, names='value')
x_scale_widget.observe(update_spectrum_plot, names='value')
y_scale_widget.observe(update_spectrum_plot, names='value')

In [ ]:
display(widgets.VBox([
    output_img,
    widgets.HBox([spectrum_type_widget, x_scale_widget, y_scale_widget])
]))
update_spectrum_plot()

### Visualizing Linear vs. Logarithmic Scales

When visualizing spectral data (amplitude or power spectrum), the choice between linear and logarithmic scales for both the frequency (x-axis) and the amplitude/power (y-axis) can significantly impact what features of the signal become apparent.

#### Y-axis Scale (Amplitude/Power):

*   **Linear Scale**: A linear y-axis emphasizes the strongest components of the signal. If there are a few very dominant frequencies, a linear scale will clearly show their magnitude relative to others. Weaker components might appear as flat lines or be completely obscured by the larger peaks.

    *   **Use case**: Best for signals where you are interested in comparing the absolute magnitudes of large peaks, or when all significant components are within a relatively narrow range of magnitudes.

*   **Logarithmic Scale**: A logarithmic y-axis compresses the range of large values and expands the range of small values. This makes it possible to visualize a wide dynamic range of power or amplitude, bringing out weaker components that would be invisible on a linear scale, while still showing the stronger ones.

    *   **Use case**: Ideal for signals with a large dynamic range, such as broadband noise (which often appears as a relatively flat line on a log scale), or when you want to identify subtle spectral peaks alongside much stronger ones. It's particularly useful in electrophysiology to see both strong neural oscillations and broadband activity.

#### X-axis Scale (Frequency):

*   **Linear Scale**: A linear x-axis provides an even distribution of frequencies across the plot. This is straightforward for inspecting specific frequency ranges.

    *   **Use case**: Useful when you are interested in a specific, narrow range of frequencies, or when you expect evenly spaced frequency components.

*   **Logarithmic Scale**: A logarithmic x-axis (often called a 'log-frequency' plot) compresses higher frequencies and expands lower frequencies. This is often more perceptually relevant for many natural signals and biological data, as changes at lower frequencies tend to be more significant in absolute terms.

    *   **Use case**: Excellent for visualizing signals where frequency content spans many orders of magnitude, or when relative frequency changes are more important than absolute ones. In neuroscience, this is common when observing phenomena across different brain wave bands (e.g., delta, theta, alpha, beta, gamma), which are logarithmically spaced.